# 04 — Model 2.1.1, typed flags as evidence (joint emission, canonical)

**The model in math terms.** The mounted M1.2 stack unchanged: the adopted Both chains per KC, the MIX2 routes, the partition link. One addition, and it is the model: per home KC per turn, the outcome is one symbol from a mutually exclusive alphabet, correct, wrong-with-flag-$j$-fired, wrong-all-quiet, with the correct-and-fired cell a **structural zero** (fires entail the home cell's failure; sixteen of seventeen fires sit on wrong home cells, the seventeenth on NA). Writing $q_l^{(j)} = P(\text{fired}_j \mid \text{wrong}, L{=}l)$:

$$P(\text{c} \mid l) = e_l(\text{c}), \qquad P(\text{w}, f_j \mid l) = e_l(\text{w})\, q_l^{(j)}, \qquad P(\text{w}, \varnothing \mid l) = e_l(\text{w}) \prod_j \big(1 - q_l^{(j)}\big)$$

Cell and flag are modeled **jointly**: the wrong's evidence is counted once, inside $e_l(\text{w})$, and the flag adds only which-kind-of-wrong. $q_1$ is pinned at 0.01 (a mastered student essentially never produces the signature blunder); $q_0 = P(\text{fired} \mid \text{wrong},\ j\ \text{presented},\ L{=}0)$ is fitted per flag from fire counts among presented wrongs, each weighted by the wrong-conditioned responsibility $P(L{=}0 \mid \text{history}, \text{wrong}) = \frac{(1-b)(1-g)}{(1-b)(1-g) + b\,s}$, the E-step weight matching the estimand's own conditioning (the flag stays out of the weight; the walk stays causal), then shrunk toward a neutral center ($\kappa_Q = 1$ toward $0.5$) and clipped to $[0.01, 0.99]$. Flag-only turns (cell NA) use the implied marginal $r_l = q_l\, e_l(\text{w})$. On flag-free turns the per-turn update is M1.2's exactly, though a trajectory can still differ from M1.2's wherever earlier flags moved the state. Prediction is untouched in-turn: a turn's flags never enter that turn's forecast, and they reach later forecasts only through the mastery state.

**Provenance, the declared revision.** The registered formulation was the factorized product, cell and flags multiplied as conditionally independent witnesses given $L$. It ran first, and its results stand as the registered record. The joint form is adopted canonically as a **theory-driven post-hoc revision**, at a measured cost of ~0.004 AUC, on measurement-fidelity grounds: the factorized form double-counts the fire-wrong pair (a fire structurally entails the home cell's failure) and prices an impossible symbol. Adopting the worse-scoring form on principle is the opposite of tuning. The factorized form is retained below as the predictive-benchmark ablation, its small edge attributed to the double-count.

**Semantics of $L$.** Post-flag, $L$ is a performance-propensity state, $P(\text{succeeding} \mid \text{knowledge and misconception history fused})$, not pure mastery; the mastery reading is recoverable only in M2.1.2's factored representation, and that fusion-versus-factoring is the content of the M2.1.1-versus-M2.1.2 fork.

**Estimation.** Two-stage as throughout: chains cached; then the $q$ tables; then $w_\text{mix}$ on its grid and the anchors on their 41-by-41 grid over the qc walks.

**File layout.**
* The model: `scripts/model_2_1_1.py` — the shared chassis `Model_2_1_1` plus the canonical `Model_2_1_1_Joint`
* The ablation arms: `scripts/model_2_1_1_ablations.py` (run in section 3; not saved)
* Comparisons, **loaded from cache, never retrained**: flag-blind M1.2 at `cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv`; classic flag-blind M1.1 at `cache/model_1_1/Model_1_1/predictions.csv`
* The inner-chain cache: `cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess/`
* Saved outputs: `cache/model_2_1_1/Model_2_1_1_Joint/` (cell at the bottom; the adopted model only)

**Protocol.** 26-fold leave-one-participant-out, predict-before-update, 312 qc targets, qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.6534.

In [ ]:
import pandas as pd
import numpy as np
from scripts.data import load_data
from scripts.evaluator import Evaluator, _metrics
from scripts.model_2_1_1 import Model_2_1_1_Joint, FLAG_HOME
from scripts.model_2_1_1_ablations import (Model_2_1_1_Factorized, Model_2_1_1_Factorized_No_Literature,
                                           Model_2_1_1_Joint_No_Shrink, Model_2_1_1_Classic_BKT)
from scripts.model_1_2_outer_chain import load_internal_chains

DATA = 'data/data_annotated.csv'
CACHE_DIR = 'cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess'
M12_PREDS = 'cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv'
M11_PREDS = 'cache/model_1_1/Model_1_1/predictions.csv'

df = load_data(DATA)
cache = load_internal_chains(CACHE_DIR)
mix2 = pd.read_csv(M12_PREDS)
m11 = pd.read_csv(M11_PREDS)
print(len(df), 'rows |', len(cache), 'cached folds |', len(mix2), 'M1.2 rows |', len(m11), 'M1.1 rows')

312 rows | 26 cached folds | 312 M1.2 rows | 312 M1.1 rows


## 1. Run
The canonical model through the shared harness with the cached inner chains. Both baselines come from stored predictions, never re-fitted.

In [ ]:
ev = Evaluator(Model_2_1_1_Joint, df, model_kwargs=dict(n_restarts=3, chain_cache=cache)).run()
preds = ev.predictions
print('done |', int(ev.metrics['n']), 'targets')

done | 312 targets


## 2. Results

### 2.1 Headline metrics against the stored baselines

In [ ]:
pd.DataFrame([dict(model='M2.1.1 (joint, canonical)', **{k: round(float(v),4) for k,v in ev.metrics.items()}),
              dict(model='M1.2 flag-blind (mounted)', **{k: round(float(v),4) for k,v in _metrics(mix2.y_true, mix2.p_pred).items()}),
              dict(model='M1.1 flag-blind (classic)', **{k: round(float(v),4) for k,v in _metrics(m11.y_true, m11.p_pred).items()})]
             ).set_index('model')[['auc','auprc_wrong','bal_acc','log_loss','accuracy','f1','n']]

,auc,auprc_wrong,bal_acc,log_loss,accuracy,f1,n
model,,,,,,,
"M2.1.1 (joint, canonical)",0.6870,0.5827,0.6182,0.6029,0.7019,0.7974,312.0
M1.2 flag-blind (mounted),0.6741,0.5747,0.6018,0.6070,0.6859,0.7860,312.0
M1.1 flag-blind (classic),0.5973,0.5164,0.5971,0.6283,0.7051,0.8099,312.0


### 2.2 Where the gain lives
The attribution split: ever-fired participants against never-fired. Note the label carefully — quiets are typed evidence too and update every participant's chains, so this split isolates who ever fired, not who was affected by the flag channel. Per-participant AUC deltas below are this run's values; they move with the calibration constants and the chain cache, so read them from the printout, not from prose.

In [ ]:
EVER = {'P01','P02','P03','P06','P11','P20','P23','P24'}
j = preds.merge(mix2, on=['participant_id','question_number'], suffixes=('_f','_b'))
for lbl, pids in (('ever-fired', EVER), ('never-fired', set(df.participant_id)-EVER)):
    sub = j[j.participant_id.isin(pids)]
    a = _metrics(sub.y_true_f, sub.p_pred_f)['auc']; b = _metrics(sub.y_true_b, sub.p_pred_b)['auc']
    print(f'{lbl:12s} n={len(sub):3d}  joint {a:.3f} | flag-blind {b:.3f} | delta {a-b:+.3f}')
rows = []
for pid, grp in j.groupby('participant_id'):
    a = _metrics(grp.y_true_f, grp.p_pred_f)['auc']; b = _metrics(grp.y_true_b, grp.p_pred_b)['auc']
    rows.append(dict(participant=pid, fired=pid in EVER, delta=round(a-b,3)))
pd.DataFrame(rows).sort_values('delta', ascending=False).set_index('participant').T

ever-fired   n= 96  joint 0.658 | flag-blind 0.636 | delta +0.022
never-fired  n=216  joint 0.651 | flag-blind 0.661 | delta -0.009


participant,P03,P24,P20,P01,P23,P05,P15,P25,P22,P21,...,P04,P26,P06,P12,P16,P09,P02,P11,P14,P17
fired,True,True,True,True,True,False,False,False,False,False,...,False,False,True,False,False,False,True,True,False,False
delta,0.111,0.094,0.083,0.074,0.037,0.031,0.0,0.0,0.0,0.0,...,0.0,0.0,-0.029,-0.037,-0.05,-0.05,-0.062,-0.094,NaN,NaN


### 2.3 Fitted q tables and bridge anchors
Fold-mean $q_0$ per flag, $P(\text{fired} \mid \text{wrong},\ \text{presented},\ L{=}0)$ under the wrong-conditioned weighting, beside the bridge. Under the neutral-center shrink the thin conditionals stay away from the saturation pathology the no-shrink arm exhibits in section 3.

In [ ]:
q = pd.DataFrame([m.q0 for m in ev.fold_models.values()]).mean().round(3)
print('fold-mean q0 (fired | wrong, unmastered):')
print(q.to_string())
print()
print('bridge fold means: s0', round(float(np.mean([m.s0 for m in ev.fold_models.values()])),3),
      '| g0', round(float(np.mean([m.g0 for m in ev.fold_models.values()])),3),
      '| w_mix', round(float(np.mean([m.shape['w_mix'] for m in ev.fold_models.values()])),3))

fold-mean q0 (fired | wrong, unmastered):
conjunction            0.126
inverse                0.498
time_axis              0.738
denominator_neglect    0.665
base_rate_neglect      0.430

bridge fold means: s0 0.08 | g0 0.105 | w_mix 0.373


### 2.4 Confusion matrices
Threshold 0.5, correct as the positive class, the canonical model beside stored M1.2. Read the FP and FN totals as nets from the printout: threshold-level movement typically comes from recovered corrects rather than newly caught wrongs, while the flags' improvement is ranking-shaped (AUC, AUPRC-wrong, log-loss).

In [ ]:
for name, p in (('M2.1.1 joint', preds), ('M1.2 flag-blind', mix2)):
    pred = (p.p_pred >= 0.5).astype(int)
    tp = int(((p.y_true == 1) & (pred == 1)).sum())
    fp = int(((p.y_true == 0) & (pred == 1)).sum())
    fn = int(((p.y_true == 1) & (pred == 0)).sum())
    tn = int(((p.y_true == 0) & (pred == 0)).sum())
    print(f'{name:16s} TP {tp:3d} | FP(wrong pred correct) {fp:3d} | FN(correct pred wrong) {fn:3d} | TN {tn:3d}')
    print(f'{"":16s} wrong-recall {tn}/{tn+fp} | correct-recall {tp}/{tp+fn}')

M2.1.1 joint     TP 183 | FP(wrong pred correct)  76 | FN(correct pred wrong)  17 | TN  36
                 wrong-recall 36/112 | correct-recall 183/200
M1.2 flag-blind  TP 180 | FP(wrong pred correct)  78 | FN(correct pred wrong)  20 | TN  34
                 wrong-recall 34/112 | correct-recall 180/200


## 3. Ablations

Four arms probing the canonical model's assumptions, each with its math. None is saved; the canonical model's save cell at the bottom is the only persistence.

**3a. Factorized emission, the registered original, now the predictive benchmark (independence added).** Cell and flags multiplied as conditionally independent witnesses given $L$:

$$P(o, F \mid L{=}l) = e_l(o) \times \prod_{j \in A_t} \begin{cases} u_l^{(j)} & f_j = \text{fired} \\ 1 - u_l^{(j)} & f_j = \text{quiet} \end{cases}$$

with $u_1$ pinned at 0.01 and $u_0^{(j)}$ the responsibility-weighted marginal fire rate shrunk toward the published written-format calibration centers ($\kappa = 5$), clipped to $[0.1, 0.9]$. The independence assumption is structurally violated here, a fire entails the home cell's failure, so on fired turns the product counts the same bad news roughly twice, a naive-Bayes overconfidence whose crashes happen to pay in a persistence-dominated cohort. Its small edge over the canonical rides the whole bundle of differences, emission form, marginal against conditional parameterization, the CALIB center against the neutral one, shrink weight five against one, so the double-count's isolated share is not identified at this n.

**3b. Factorized, no-literature.** Arm 3a with the calibration deleted: $u_0$ = the weighted fire rate alone, $\kappa = 0$, clip floor 0.01. Channels with no training fires collapse to the floor and go inert.

**3c. Joint, no-shrink.** The canonical form with its neutral-center shrink deleted, $q_0$ $\kappa$-free at the 0.01 floor. The thin-conditional pathology arm: $q_0$ divides by wrongs only, and unshrunk it saturates (time-axis toward 0.99), flipping wrong-without-fire into evidence of mastery.

**3d. Classic BKT plus flags (on Model_1_1).** The user's Model_1_1 chassis unchanged, chains trained on pseudo-turns (the qc label broadcast to every designed KC), prediction the compensatory mean, the per-cell annotation unused by design. The joint alphabet needs the home cell and this chassis has none, so the integration is necessarily the marginal-factor form, each presented flag multiplying its typed factor into its home chain's update during the walk, off-designed homes updated flag-only. Against flag-blind Model_1_1 this asks whether the typed channel travels to a chassis that never sees cells.

In [ ]:
kwa = dict(n_restarts=3, chain_cache=cache)
arms = {}
for cls in (Model_2_1_1_Factorized, Model_2_1_1_Factorized_No_Literature, Model_2_1_1_Joint_No_Shrink):
    arms[cls.__name__] = Evaluator(cls, df, model_kwargs=kwa).run()
    print(cls.__name__, 'done')
arms['Model_2_1_1_Classic_BKT'] = Evaluator(Model_2_1_1_Classic_BKT, df, model_kwargs=dict(n_restarts=3)).run()
print('Model_2_1_1_Classic_BKT done')

Model_2_1_1_Factorized done
Model_2_1_1_Factorized_No_Literature done
Model_2_1_1_Joint_No_Shrink done
Model_2_1_1_Classic_BKT done


In [ ]:
labels = {'Model_2_1_1_Factorized': 'Factorized, calibrated (independence; predictive benchmark)',
          'Model_2_1_1_Factorized_No_Literature': 'Factorized, no-literature',
          'Model_2_1_1_Joint_No_Shrink': 'Joint, no-shrink',
          'Model_2_1_1_Classic_BKT': 'Classic BKT + flags (Model_1_1 chassis)'}
rows = [dict(model='M2.1.1 (joint, canonical)', **{k: round(float(v),4) for k,v in ev.metrics.items()})]
for name, e in arms.items():
    rows.append(dict(model=labels[name], **{k: round(float(v),4) for k,v in e.metrics.items()}))
rows.append(dict(model='M1.2 flag-blind (mounted ref)', **{k: round(float(v),4) for k,v in _metrics(mix2.y_true, mix2.p_pred).items()}))
rows.append(dict(model='M1.1 flag-blind (classic ref)', **{k: round(float(v),4) for k,v in _metrics(m11.y_true, m11.p_pred).items()}))
pd.DataFrame(rows).set_index('model')[['auc','auprc_wrong','bal_acc','log_loss','accuracy']]

,auc,auprc_wrong,bal_acc,log_loss,accuracy
model,,,,,
"M2.1.1 (joint, canonical)",0.6870,0.5827,0.6182,0.6029,0.7019
"Factorized, calibrated (independence; predictive benchmark)",0.6907,0.5907,0.6029,0.6003,0.6923
"Factorized, no-literature",0.6867,0.5858,0.6138,0.6031,0.6987
"Joint, no-shrink",0.6837,0.5779,0.6207,0.6051,0.7051
Classic BKT + flags (Model_1_1 chassis),0.6103,0.5323,0.6184,0.6221,0.7147
M1.2 flag-blind (mounted ref),0.6741,0.5747,0.6018,0.6070,0.6859
M1.1 flag-blind (classic ref),0.5973,0.5164,0.5971,0.6283,0.7051


**Observations.**
* **The factorized benchmark's small edge rides a bundle, not one mechanism.** It scores ~0.004 AUC above the canonical, and the arms differ in emission form, parameterization (marginal against conditional), prior center, and shrink weight together, so the double-count's isolated share is not identified at this n. It is retained as the predictive benchmark, not the measurement model; at larger n I hypothesize the gap closes from the other side as the joint form's conditionals stop starving, a hypothesis, not a result.
* **The deletions are roughly additive and the primary contrast survives the harshest cell.** Independence ~0.004, literature ~0.004, both ~0.0075; joint-no-lit still clears flag-blind M1.2. The headline claim rests on the flags, not the priors and not the emission form.
* **The literature buys the thin channels, not the performance.** Without calibration, conjunction and inverse collapse to the floor and go inert while the pooled result barely moves.
* **The shrink is load-bearing for the canonical form specifically.** Unshrunk, the joint $q_0$ saturates on its wrong-only denominators (time-axis toward 0.99), inverting wrong-without-fire into mastery evidence; the faithful form needs its priors more than the factorized form needs its calibration.
* **The mechanism travels.** On the classic chassis that never sees cells, the flags still add over flag-blind Model_1_1, the gain concentrated in ever-fired participants: the typed channel is not an artifact of the mounted stack or the cell annotation.
* Approximate values above are environment-dependent (calibration constants, chain cache); read exact figures from this run's tables.

## 4. Conclusion

* **The typed evidence channel earns its keep on the canonical joint form.** Roughly +0.013 AUC over flag-blind M1.2 with log-loss improving, the gain concentrated where the mechanism operates (ever-fired participants, post-fire turns), never-fired participants essentially flat.
* **The claim is measurement-faithful.** The canonical emission encodes the instrument, fires as subtypes of wrong, the impossible correct-and-fired symbol a structural zero, no independence assumption; the registered factorized form survives as the predictive benchmark with its edge explained, not celebrated.
* **Post-flag L is a performance-propensity state,** knowledge and misconception history fused; the mastery reading is recoverable only in M2.1.2's factored representation, which is the fork's content.
* **Robustness.** The gain survives every ablation cell including both deletions at once. Q_PIN_1 and Q_KAPPA have not yet received their own sensitivity sweeps, the earlier sweeps addressed the factorized marginal parameterization, so I treat their values as stated assumptions pending direct runs.
* **Caveats.** n = 312 qc targets from 26 participants; participant-clustered intervals pending (a paired preview interval straddles zero, so the honest register is promising, not decisive); per-participant values are environment-dependent and reported from outputs, not prose.

## 5. Save
Persist the adopted model only, the ablation arms are not saved. Per-fold bridge, shape, and q-tables, plus predictions, metrics, and the index.

In [ ]:
import os
import json
from scripts.model_2_1_1 import save_model_2_1_1_from_evaluator

out_dir = 'cache/model_2_1_1'
save_model_2_1_1_from_evaluator(ev, out_dir)

'cache/model_2_1_1/Model_2_1_1_Joint'